# 📌 Topic 5: Convolutional Neural Networks (CNNs)
> **Deep Learning Crash Course — Part 5**

In this notebook, we will explore **Computer Vision with CNNs**:
1. Why flat MLPs fail on images (losing spatial structure).
2. **Convolution Operation**: Kernels, Filters, Stride, and Padding ($O = \lfloor \frac{W-K+2P}{S} \rfloor + 1$).
3. **Pooling Layers** (Max Pooling) & Feature Maps.
4. PyTorch CNN Implementation & Visualizing learned filters.


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


### 5.1 CNN Architecture in PyTorch


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()
        # Conv Block 1
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Conv Block 2
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        
        # Classifier Head
        self.fc = nn.Linear(32 * 7 * 7, num_classes)
        
    def forward(self, x):
        # Input shape: [B, 1, 28, 28]
        feat1 = self.pool(self.relu(self.conv1(x)))  # -> [B, 16, 14, 14]
        feat2 = self.pool(self.relu(self.conv2(feat1))) # -> [B, 32, 7, 7]
        
        flat = feat2.view(feat2.size(0), -1)
        out = self.fc(flat)
        return out, feat1, feat2

# Create Synthetic Image Tensor [Batch=8, Channels=1, Height=28, Width=28]
images = torch.randn(8, 1, 28, 28).to(device)
model = SimpleCNN().to(device)

output, f1, f2 = model(images)

print(f"Input Image Shape: {images.shape}")
print(f"Conv Layer 1 Feature Map Output: {f1.shape}")
print(f"Conv Layer 2 Feature Map Output: {f2.shape}")
print(f"Final Class Logits Output: {output.shape}")

# Visualize Feature Maps
sample_f1 = f1[0].detach().cpu().numpy()

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for idx, ax in enumerate(axes.flat):
    ax.imshow(sample_f1[idx], cmap="magma")
    ax.axis("off")
    ax.set_title(f"F{idx+1}", fontsize=10)
plt.suptitle("CNN 1st Conv Layer Feature Maps", fontsize=14)
plt.tight_layout()
plt.show()
